In [ ]:
import numpy as np
from sklearn.metrics import silhouette_samples

def lagrangepoly(X, Y):
    """
    Fits a Lagrange interpolation polynomial to (Threshold, Silhouette) pairs.

    Parameters:
    X : array-like, shape (M,) - The threshold values (\delta_m)
    Y : array-like, shape (M,) - The silhouette values (S_m)

    Returns:
    P : ndarray - The coefficients of the Lagrange polynomial.
    R : ndarray - The roots of the derivative of the polynomial (extrema).
    S : ndarray - The polynomial evaluated at the extrema.
    """
    X = np.asarray(X, dtype=float)
    Y = np.asarray(Y, dtype=float)
    N = len(X)

    # Initialize matrix to store basis polynomials
    pvals = np.zeros((N, N))

    for i in range(N):
        # Roots for the basis polynomial are all X except X[i]
        X_ex = np.concatenate((X[:i], X[i+1:]))
        pp = np.poly(X_ex)

        # Normalize the polynomial so it evaluates to 1 at X[i]
        val = np.polyval(pp, X[i])
        if abs(val) < 1e-12:
            val = 1e-12  # Prevent division by zero if thresholds overlap

        pvals[i, :] = pp / val

    # The final polynomial is the weighted sum of basis polynomials
    P = np.dot(Y, pvals)

    # Calculate derivative to find extrema (peaks/valleys)
    dP = np.polyder(P)
    R = np.roots(dP)
    S = np.polyval(P, R)

    return P, R, S

def slht(s, idx, n, m_counts, nk):
    """
    Computes per-cluster silhouette averages and the Global Silhouette Index (GSI).

    Parameters:
    s : ndarray - Raw silhouette samples for every data point.
    idx : ndarray - Cluster assignment labels.
    n : int - Total number of data points.
    m_counts : ndarray - Number of points assigned in the initial radius thresholding.
    nk : int - Number of clusters (k).

    Returns:
    S : ndarray - The average silhouette value for each cluster.
    GSI : float - The Global Silhouette Index.
    """
    S = np.zeros(nk)
    for i in range(nk):
        cluster_silhouettes = s[idx == i]
        if len(cluster_silhouettes) > 0:
            S[i] = np.mean(cluster_silhouettes)

    GSI = np.mean(S)
    return S, GSI

def soc(x, nk, factor):
    """
    The core Mountain Clustering engine, implementing SOC and IMC logic.

    Parameters:
    x : ndarray, shape (n, 3) - The flattened image feature vectors (RGB).
    nk : int - The desired number of clusters.
    factor : ndarray or float - The \beta_m scaling factors for the threshold.

    Returns:
    dict : Contains cluster assignments ('idx'), normalized centers ('cc_norm'),
           thresholds ('dl'), assignment counts ('m'), and total points ('n').
    """
    n, _ = x.shape

    # 1. Normalize data to a unit hypercube (Scale all features to [0, 1])
    x_min = np.min(x, axis=0)
    x_max = np.max(x, axis=0)
    # Add epsilon to prevent division by zero for solid colors
    u = (x - x_min) / (x_max - x_min + 1e-10)

    idx = np.full(n, -1, dtype=int)
    unassigned = np.ones(n, dtype=bool)

    dl = np.zeros(nk)
    cc_norm = np.zeros((nk, 3))
    m_counts = np.zeros(nk, dtype=int)

    # Ensure factor is an array so each cluster can have a distinct threshold
    if np.isscalar(factor):
        factor = np.full(nk, factor, dtype=float)

    for v in range(nk):
        curr_indices = np.where(unassigned)[0]
        curr_pts = u[curr_indices]
        num_curr = len(curr_pts)

        if num_curr == 0:
            break

        # 2. Compute Neighborhood Threshold (\delta_m)
        min_x = np.min(curr_pts, axis=1)
        sum_x = np.sum(curr_pts, axis=1) + 1e-10
        d_val = np.sum(min_x / sum_x)
        dl[v] = (1.0 / (2 * num_curr)) * d_val * factor[v]

        # Prevent zero-radius clusters
        if dl[v] < 1e-4:
            dl[v] = 1e-4

        # 3. Compute Mountain Potential using Batching (Prevents Out-Of-Memory errors)
        P = np.zeros(num_curr)
        batch_size = 2000 # Process 2000 pixels at a time

        for i in range(0, num_curr, batch_size):
            end = min(i + batch_size, num_curr)
            # Calculate squared Euclidean distances for the batch
            diff = curr_pts[i:end, np.newaxis, :] - curr_pts[np.newaxis, :, :]
            dist_sq = np.sum(diff ** 2, axis=2)
            # Apply Gaussian mountain function
            P[i:end] = np.sum(np.exp(-dist_sq / (dl[v]**2)), axis=1)

        # 4. Select Center and Form Cluster
        center_idx_in_curr = np.argmax(P)
        cc_norm[v] = curr_pts[center_idx_in_curr]

        # 5. Assign points within the threshold radius and remove them from working set
        dists_to_center = np.sum((curr_pts - cc_norm[v])**2, axis=1)
        assigned_in_curr = dists_to_center <= (dl[v]**2)

        assigned_orig = curr_indices[assigned_in_curr]
        idx[assigned_orig] = v
        unassigned[assigned_orig] = False
        m_counts[v] = len(assigned_orig)

    # 6. Distribute leftover points to the nearest cluster center
    leftover_indices = np.where(unassigned)[0]
    if len(leftover_indices) > 0:
        leftover_pts = u[leftover_indices]
        valid_centers = cc_norm[:nk] # Only compare against successfully formed centers

        # Compute distances to all valid centers
        dist_to_centers = np.sum((leftover_pts[:, np.newaxis, :] - valid_centers[np.newaxis, :, :])**2, axis=2)
        best_centers = np.argmin(dist_to_centers, axis=1)
        idx[leftover_indices] = best_centers

    return {'idx': idx, 'cc_norm': cc_norm, 'dl': dl, 'm': m_counts, 'n': n}

def imc2(x, nk, base_factor):
    """
    Wrapper function to run IMC-1 and IMC-2 baselines.
    IMC-1 passes base_factor = 1.0
    IMC-2 passes base_factor = nk / (nk + 1.0)
    """
    return soc(x, nk, factor=base_factor)

def factorcal(x, nk, iter_max=10):
    """
    Iterative optimization loop for SOC.
    Uses Lagrange interpolation to find the optimal \beta_m factors that maximize GSI.

    Parameters:
    x : ndarray - Flattened image data.
    nk : int - Number of clusters.
    iter_max : int - Maximum number of optimization iterations.

    Returns:
    ndarray - The optimal \beta factors array (length nk) that achieved the highest GSI.
    """
    factors = np.ones((iter_max + 1, nk))
    GSI_history = np.zeros(iter_max)

    for i in range(iter_max):
        print(f"   -> SOC Iteration {i+1}/{iter_max}...")

        # Run clustering with current factors
        result = soc(x, nk, factors[i])

        # Evaluate performance
        s = silhouette_samples(x, result['idx'])
        S, GSI_history[i] = slht(s, result['idx'], result['n'], result['m'], nk)

        # To perform Lagrange interpolation, thresholds (\delta) must be unique.
        # If clusters collapse and share thresholds, we break to avoid division by zero.
        if len(np.unique(result['dl'])) < nk:
            print("      [!] Non-unique thresholds detected. Halting optimization.")
            break

        # Build Lagrange polynomial relating Thresholds (dl) to Silhouettes (S)
        P, r_roots, _ = lagrangepoly(result['dl'], S)

        # We want to find the threshold (\eta) where Silhouette theoretically equals 1.0 (Maximum)
        polym = P.copy()
        polym[-1] -= 1.0

        # Find roots of the shifted polynomial
        roots = np.roots(polym)
        real_roots = roots[np.isreal(roots)].real

        if len(real_roots) == 0:
            # Fallback if no real roots: use the current mean threshold
            eta = np.mean(result['dl'])
        else:
            # Select the root that evaluates closest to S=1 on the original polynomial
            evaluations = np.abs(np.polyval(P, real_roots) - 1.0)
            best_idx = np.argmin(evaluations)
            eta = abs(real_roots[best_idx])

        # Update factors for the next iteration: \beta_m = \eta / \delta_m
        factors[i+1] = eta / (result['dl'] + 1e-10)

    # Find the iteration that achieved the absolute maximum GSI
    best_iteration = np.argmax(GSI_history)
    print(f"   -> Optimal factors found at iteration {best_iteration + 1} (GSI: {GSI_history[best_iteration]:.4f})")

    return factors[best_iteration]